In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("supply-chain-data-platform")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/29 16:06:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pathlib import Path

data_dir = Path("../data/raw")
parquet_files = sorted(data_dir.glob("*.parquet"))

parquet_files

[PosixPath('../data/raw/customer_first.parquet'),
 PosixPath('../data/raw/customers_second.parquet'),
 PosixPath('../data/raw/orders_first.parquet'),
 PosixPath('../data/raw/orders_second.parquet'),
 PosixPath('../data/raw/products_first.parquet'),
 PosixPath('../data/raw/products_second.parquet'),
 PosixPath('../data/raw/regions.parquet')]

In [4]:
for file in parquet_files:
    print("=" * 80)
    print(f"File: {file.name}")
    
    df = spark.read.parquet(str(file))
    
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")
    print("Schema:")
    df.printSchema()
    
    print("Sample:")
    df.show(5, truncate=False)

File: customer_first.parquet
Rows: 1990
Columns: 6
Schema:
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)

Sample:
+-----------+----------+---------+------------------------+------------------+-----+
|customer_id|first_name|last_name|email                   |city              |state|
+-----------+----------+---------+------------------------+------------------+-----+
|C00001     |Emily     |Mooney   |rushjeff@ryan.org       |Johnsonmouth      |MS   |
|C00002     |Andrea    |Sellers  |mccoykiara@kelly.com    |Stephenfort       |WY   |
|C00003     |Craig     |Hayes    |rebeccamiller@yahoo.com |South Stephenshire|LA   |
|C00004     |Bryan     |Scott    |lawrence05@campbell.info|Chrisland         |ND   |
|C00005     |Sean      |Vasquez  |carrie45@yahoo.com      |East Dennistown   |RI   |
+----------

In [5]:
from pyspark.sql.functions import col, count, isnan, when

summary = []

for file in parquet_files:
    df = spark.read.parquet(str(file))
    summary.append({
        "file": file.name,
        "rows": df.count(),
        "columns": len(df.columns),
        "column_names": df.columns,
    })

summary

[{'file': 'customer_first.parquet',
  'rows': 1990,
  'columns': 6,
  'column_names': ['customer_id',
   'first_name',
   'last_name',
   'email',
   'city',
   'state']},
 {'file': 'customers_second.parquet',
  'rows': 10,
  'columns': 6,
  'column_names': ['customer_id',
   'first_name',
   'last_name',
   'email',
   'city',
   'state']},
 {'file': 'orders_first.parquet',
  'rows': 9990,
  'columns': 6,
  'column_names': ['order_id',
   'customer_id',
   'product_id',
   'order_date',
   'quantity',
   'total_amount']},
 {'file': 'orders_second.parquet',
  'rows': 10,
  'columns': 6,
  'column_names': ['order_id',
   'customer_id',
   'product_id',
   'order_date',
   'quantity',
   'total_amount']},
 {'file': 'products_first.parquet',
  'rows': 490,
  'columns': 5,
  'column_names': ['product_id',
   'product_name',
   'category',
   'brand',
   'price']},
 {'file': 'products_second.parquet',
  'rows': 10,
  'columns': 5,
  'column_names': ['product_id',
   'product_name',
   'cate

In [6]:
import pandas as pd

pd.DataFrame(summary)

,file,rows,columns,column_names
0,customer_first.parquet,1990,6,"[customer_id, first_name, last_name, email, ci..."
1,customers_second.parquet,10,6,"[customer_id, first_name, last_name, email, ci..."
2,orders_first.parquet,9990,6,"[order_id, customer_id, product_id, order_date..."
3,orders_second.parquet,10,6,"[order_id, customer_id, product_id, order_date..."
4,products_first.parquet,490,5,"[product_id, product_name, category, brand, pr..."
5,products_second.parquet,10,5,"[product_id, product_name, category, brand, pr..."
6,regions.parquet,4,2,"[region_id, region]"
